New Notebook Created by Jupyter MCP Server

In [1]:
import os, inspect
from pathlib import Path
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str((Path.cwd() / 'benchmark/projects').resolve())
from ai_data_scientist import project_manager as pm, lifecycle, language_router
print('cwd:', Path.cwd())
print('resolve:', inspect.signature(pm.resolve_project))
print('enqueue:', inspect.signature(pm.enqueue_write))
print('lifecycle:', [(n, str(inspect.signature(getattr(lifecycle,n)))) for n in ['register_run','mark_execution_start','mark_execution_end','mark_write_start','mark_write_end','mark_completed','mark_failed']])
handle = pm.resolve_project('kaggle-v020-kaggle-exp-01-titanic')
pm.ensure_notebook(handle)
print('notebook:', handle.notebook_path)
print('language:', language_router.detect_language('生存者と非生存者の違いを分析してください'))

cwd: /home/nahisaho/GitHub/jupytermind/benchmark
resolve: (name: 'str', projects_root: 'Path | str | None' = None) -> 'ProjectHandle'
enqueue: (handle: 'ProjectHandle', cell_mutation) -> 'Path'
lifecycle: [('register_run', "(run_id: 'str', notebook_path: 'Path | str') -> 'None'"), ('mark_execution_start', "(run_id: 'str') -> 'None'"), ('mark_execution_end', "(run_id: 'str') -> 'None'"), ('mark_write_start', "(run_id: 'str') -> 'None'"), ('mark_write_end', "(run_id: 'str') -> 'None'"), ('mark_completed', "(run_id: 'str') -> 'None'"), ('mark_failed', "(run_id: 'str') -> 'None'")]
notebook: /home/nahisaho/GitHub/jupytermind/benchmark/benchmark/projects/kaggle-v020-kaggle-exp-01-titanic/notebooks/kaggle-v020-kaggle-exp-01-titanic.ipynb
language: ja


In [2]:
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = '/home/nahisaho/GitHub/jupytermind/benchmark/projects'
handle = pm.resolve_project('kaggle-v020-kaggle-exp-01-titanic', projects_root=os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'])
pm.ensure_notebook(handle)
print(handle.notebook_path)

/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-01-titanic/notebooks/kaggle-v020-kaggle-exp-01-titanic.ipynb


In [3]:
import nbformat, textwrap
from ai_data_scientist import lifecycle
from dataclasses import asdict
cells = []
def md(source):
    cells.append(nbformat.v4.new_markdown_cell(textwrap.dedent(source).strip()))
def code(source):
    cells.append(nbformat.v4.new_code_cell(textwrap.dedent(source).strip()))
md('''
# Titanic：生存者と非生存者の比較
**run_id:** `v020-exp-01` / **言語:** 日本語

Kaggle参照先：https://www.kaggle.com/competitions/titanic 。AI Data Scientist Skill と Jupyter MCP による記述・関連分析。予測モデルの競技スコアや因果効果は目的としない。

実行とAPI取得はすべてJupyter MCP。構成・Markdown・根拠manifestの編集は `project_manager.enqueue_write`、コードセルの実行結果保存はJupyter MCPが担当する。認証トークンは表示・保存しない。`mcp_gateway.run_and_record` はこの対話で注入可能なPython側MCP transportがないため使用しない（二重実行や偽クライアントを作らず、接続済みMCPのexecute_cellを使用）。外部CSVミラーを使用しない。
''')
code('''
import os, json, hashlib, io, contextlib, math
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
from contextlib import contextmanager
import numpy as np
import pandas as pd
import nbformat
from IPython.display import display
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from ai_data_scientist import project_manager as pm, lifecycle, language_router
from ai_data_scientist import ingestion, cleaning, eda, stats_analysis, visualization
from ai_data_scientist import data_definition, analysis_assumptions, data_quality, sensitivity, notebook_audit

PROJECT = 'kaggle-v020-kaggle-exp-01-titanic'
RUN_ID = 'v020-exp-01'
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = '/home/nahisaho/GitHub/jupytermind/benchmark/projects'
handle = pm.resolve_project(PROJECT, projects_root=os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'])
lifecycle.register_run(RUN_ID, handle.notebook_path)
phase_events = []

def event(name):
    phase_events.append({'phase': name, 'at_utc': datetime.now(timezone.utc).isoformat()})

@contextmanager
def execution_phase(name):
    if lifecycle.is_cancel_requested(RUN_ID):
        lifecycle.mark_failed(RUN_ID)
        raise RuntimeError('協調的中止が要求されました。')
    lifecycle.mark_execution_start(RUN_ID)
    event('execution_start:' + name)
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        event('failed:' + name)
        raise
    finally:
        lifecycle.mark_execution_end(RUN_ID)
        event('execution_end:' + name)

@contextmanager
def write_phase(name):
    lifecycle.mark_write_start(RUN_ID)
    event('write_start:' + name)
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        raise
    finally:
        lifecycle.mark_write_end(RUN_ID)
        event('write_end:' + name)

def chart(table, x, y, title, xlabel, ylabel):
    png = visualization.render_chart(table, kind='bar', x=x, y=y, title=title, xlabel=xlabel, ylabel=ylabel)
    output = visualization.build_image_output(png)
    with write_phase('chart:' + title):
        display(output.data, raw=True, metadata=output.metadata)

print('言語:', language_router.detect_language('Titanicの生存者と非生存者の違いを調べてください'))
print('Notebook:', handle.notebook_path)
print('lifecycle:', asdict(lifecycle.get_run_status(RUN_ID)))
''')
md('''
## 分析計画（データ比較前に固定）
1. APIで公開データを検索し、原形の学習用CSVを含む候補を選ぶ。構造・生存ラベルを検査し、slug・ファイル・取得日時・SHA-256を保存する。テストデータ・gender_submissionは真の生存ラベルとして使わない。
2. データ定義と分析前提をmanifestに記録する。範囲、カテゴリ、必須列、ID一意性、重複・欠損を検査する。年齢不明を捨てず、群別人数とともに示す。
3. 性別、客室等級、年齢層、運賃四分位、同行家族、乗船港、年齢欠損、客室情報有無を比較する。生存率とWilson 95%区間、人数・生存数を表で示す。年齢・運賃は中央値/IQR・平均・Mann–Whitneyの効果量を併記する。率を主たる比較尺度にし、オッズ比は率差と同一視しない。
4. 可視化は日本語の率棒グラフを採用する。棒は点推定のみで、区間は対応表に掲載。年齢欠損を独立カテゴリとして可視化し、ID・氏名・チケット番号を数値尺度として扱わない。
5. 探索的検定は10比較にHolm補正をする。信頼区間はモデル上の不確実性を示し、無作為抽出や乗客間独立が保証されるわけではない。
6. 初回結果を読み、結論を変えうる未解決点だけを自然言語で追加依頼し、最大3反復。交絡、年齢欠損、同行者の依存性などを優先する。反復を探索的と明記し、因果主張はしない。
7. 新しいカーネルで全コードセルを上から実行し、visual_audit=Trueのnotebook_audit、画像メタデータ、根拠の解決、lifecycle completedを確認する。失敗時は成功として扱わない。
''')
code('''
with execution_phase('Kaggle公開データ検索・取得・検証'):
    from kaggle.api.kaggle_api_extended import KaggleApi
    DATASET_SLUG = 'rahulsah06/titanic'
    DATA_FILE = 'train.csv'
    EXPECTED_SHA256 = '7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f'
    data_dir = pm.ensure_data_dir(handle)
    csv_path = data_dir / DATA_FILE
    provenance_path = data_dir / 'retrieval-manifest.json'
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api = KaggleApi()
            api.authenticate()
            candidates = []
            for page in range(1, 4):
                found = api.dataset_list(search='titanic', file_type='csv', page=page) or []
                candidates.extend({'ref': d.ref, 'title': d.title} for d in found)
                if any(d['ref'] == DATASET_SLUG for d in candidates):
                    break
            if not any(d['ref'] == DATASET_SLUG for d in candidates):
                raise RuntimeError('対応する公開データがAPI検索で見つかりません。')
            file_inventory = {}
            for slug in ['heptapod/titanic', 'yasserh/titanic-dataset', DATASET_SLUG]:
                listing = api.dataset_list_files(slug)
                file_inventory[slug] = [f.name for f in listing.files]
            if DATA_FILE not in file_inventory[DATASET_SLUG]:
                raise RuntimeError('候補データに必要な学習CSVがありません。')
            if not csv_path.exists():
                api.dataset_download_file(DATASET_SLUG, DATA_FILE, path=str(data_dir), quiet=True)
                if not csv_path.exists():
                    import zipfile
                    with zipfile.ZipFile(data_dir / (DATA_FILE + '.zip')) as archive:
                        csv_path.write_bytes(archive.read(DATA_FILE))
                retrieval_manifest = {
                    'run_id': RUN_ID,
                    'competition_reference': 'https://www.kaggle.com/competitions/titanic',
                    'owner_dataset_slug': DATASET_SLUG,
                    'filename': DATA_FILE,
                    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                    'sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(),
                    'source_kind': 'Kaggle公開データ（競技データの再配布候補、公式とのバイト一致は未確認）'
                }
                with write_phase('retrieval_manifest'):
                    provenance_path.write_text(json.dumps(retrieval_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
            else:
                if not provenance_path.exists():
                    raise RuntimeError('キャッシュに取得manifestがありません。由来不明のデータは使用しません。')
                retrieval_manifest = json.loads(provenance_path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('Kaggle API/由来検証の失敗種別:', type(exc).__name__)
        raise RuntimeError('Kaggle API取得・検証を完了できません。外部ソースにフォールバックせず停止します。認証情報保護のため例外本文は表示しません。') from None
    actual_sha = hashlib.sha256(csv_path.read_bytes()).hexdigest()
    assert actual_sha == retrieval_manifest['sha256'] == EXPECTED_SHA256, '保存済みCSVのSHA-256が変更されています。'
    assert retrieval_manifest['owner_dataset_slug'] == DATASET_SLUG and retrieval_manifest['filename'] == DATA_FILE
    loaded = ingestion.ingest(ingestion.SourceSpec(kind='csv', location=str(csv_path)))
    df_raw = loaded.dataframe
    required = {'PassengerId','Survived','Pclass','Name','Sex','Age','SibSp','Parch','Ticket','Fare','Cabin','Embarked'}
    assert set(df_raw.columns) == required and len(df_raw) == 891 and not loaded.truncated
    assert df_raw['Survived'].value_counts().to_dict() == {0:549, 1:342}
    print('検索候補:', json.dumps(candidates, ensure_ascii=False))
    print('APIファイル一覧:', json.dumps(file_inventory, ensure_ascii=False))
    print('選定理由: 原形のtrain.csv/test.csvを分離して持つ公開データを優先。結合済み・cleanedデータと推測ラベルを避ける。')
    print('取得manifest:', json.dumps(retrieval_manifest, ensure_ascii=False, indent=2))
    print('API再確認日時UTC:', datetime.now(timezone.utc).isoformat())
    print('再実行では検証済みキャッシュを使用。取得日は初回のまま維持し、SHAを毎回検査。')
    print('標準学習データと形状・ラベル集計が一致。ただし公式競技CSVとのバイト一致を保証しない。')
''')
md('''
## データ定義・前提・意味的異常検査
再配布データの説明にデータ辞書がないため、単位・意味の確認状態を誇張しない。観測したカテゴリ・値域の一致は「意味の検証」ではない。客室等級、年齢、運賃、同行者の意味は標準的なTitanic列名からの推定として扱う。特に運賃は1人当たり価格と断言できず、通貨単位も未検証。CabinKnownは居住区画の優劣ではなく記録の有無にすぎない。

独立した参照データは与えられていない。同じ競技データの別再配布は独立標本ではないので、validate_anomalies・compare_datasetsによる外部妥当性検証は適用しない。極端に高い運賃・幼児の小数年齢は即エラーとはせず、意味的範囲チェックと感度分析を使う。
''')
code('''
with execution_phase('manifest・品質検査・前処理'):
    F = data_definition.FieldValue
    definitions = {
        'PassengerId': ('レコードの乗客識別子', '識別子'),
        'Survived': ('生存ラベル（0=非生存、1=生存）', '二値'),
        'Pclass': ('客室等級（1/2/3）', 'カテゴリ'),
        'Sex': ('データに記録された性別', 'カテゴリ'),
        'Age': ('乗客年齢、幼児は小数を含む', '年と推定'),
        'SibSp': ('同乗きょうだい・配偶者数と推定', '人数'),
        'Parch': ('同乗親・子数と推定', '人数'),
        'Fare': ('記録された運賃、個人料金か共同チケット料金か未確定', None),
        'Embarked': ('乗船港コード（C/Q/S）', 'カテゴリ'),
        'Cabin': ('客室情報。欠損は不明であり無客室とは解釈しない', '文字列'),
        'Ticket': ('チケット識別子。同一番号の乗客は独立でない可能性', '識別子'),
        'Name': ('氏名。個人の表示・ランキングには使用しない', '文字列'),
        'FamilySize': ('SibSp + Parch + 1。本データ上の同行家族人数指標', '人数')
    }
    definition_manifest = data_definition.build_manifest(
        source={'value': DATASET_SLUG, 'status': 'verified', 'sha256': actual_sha},
        dataset_scope={'value': 'train.csvの891レコード。全乗客や無作為標本ではない', 'status': 'verified'},
        variables={name: {'meaning': F(meaning, 'inferred', '列名・値域と一般的なTitanic定義。再配布辞書は未確認'),
                          'unit': F(unit, 'unknown' if unit is None else 'inferred')}
                   for name, (meaning, unit) in definitions.items()},
        transformations=('重複検査（除去0件）', 'FamilySize=SibSp+Parch+1', '年齢不明を別群', 'Fareの標本内四分位', '客室記録有無')
    )
    print('データ定義manifest:', json.dumps(asdict(definition_manifest), ensure_ascii=False, default=str))
    print('未解決定義:', definition_manifest.unresolved_fields())
    assumptions_manifest = analysis_assumptions.AnalysisAssumptionManifest(
        analysis_scope={'target':'取得した学習レコードにおける生存との関連'},
        causal_scope='associational',
        sampling={'design':'競技の学習標本、抽出機構は不明'},
        assumptions=(
            analysis_assumptions.Assumption('label-meaning', 'Survivedの意味・符号化が標準Titanicと同じ', 'assumed', conclusion_critical=True, impact_if_false='生存に関する全解釈が変わる'),
            analysis_assumptions.Assumption('source-correspondence', '公開再配布が競技train.csvに対応する', 'assumed', conclusion_critical=True, impact_if_false='競技データについての一般化を制限'),
            analysis_assumptions.Assumption('descriptive-only', '条件付き関連を原因と解釈しない', 'verified'),
            analysis_assumptions.Assumption('age-missing', '年齢欠損はランダムとは仮定せず別群・感度分析で検討', 'assumed', impact_if_false='子供・成人比較の一般化を制限'),
            analysis_assumptions.Assumption('independent-ci', '初回のWilson区間・検定は乗客独立の近似', 'assumed', impact_if_false='区間・p値が過度に精密になる可能性')
        )
    )
    print('分析前提manifest:', json.dumps(asdict(assumptions_manifest), ensure_ascii=False))
    print('前提検査finding:', [asdict(f) for f in analysis_assumptions.check_manifest(assumptions_manifest)])
    schema = {
        'PassengerId': {'not_null':True, 'unique':True, 'min':1},
        'Survived': {'not_null':True, 'allowed':[0,1]},
        'Pclass': {'not_null':True, 'allowed':[1,2,3]},
        'Sex': {'not_null':True, 'allowed':['male','female']},
        'Age': {'min':0, 'max':100},
        'Fare': {'min':0, 'not_null':True},
        'SibSp': {'min':0, 'max':20, 'not_null':True},
        'Parch': {'min':0, 'max':20, 'not_null':True},
        'Embarked': {'allowed':['C','Q','S']}
    }
    semantic_anomalies = data_quality.detect_anomalies(df_raw, schema=schema)
    print('意味的異常:', [asdict(a) for a in semantic_anomalies])
    assert not semantic_anomalies, '意味的異常が見つかったため分析前に確認が必要です。'
    assert all((df_raw[c].dropna() % 1 == 0).all() for c in ['PassengerId','Pclass','SibSp','Parch','Survived'])
    cleaned = cleaning.clean_dataset(df_raw, operation='drop_duplicates')
    print('クリーニング:', {k:v for k,v in vars(cleaned).items() if not isinstance(v,pd.DataFrame)})
    df = next(v for v in vars(cleaned).values() if isinstance(v,pd.DataFrame)).copy()
    assert len(df) == len(df_raw), '予期しない重複がありました。'
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['FamilyGroup'] = pd.cut(df['FamilySize'], bins=[0,1,4,20], labels=['1人','2–4人','5人以上']).astype(str)
    df['AgeGroup'] = pd.cut(df['Age'], bins=[-0.01,12,18,35,60,100], right=False,
                            labels=['0–11歳','12–17歳','18–34歳','35–59歳','60歳以上']).astype(object).fillna('年齢不明')
    df['AgeMissing'] = df['Age'].isna().map({True:'年齢不明',False:'年齢あり'})
    df['CabinKnown'] = df['Cabin'].notna().map({True:'客室記録あり',False:'客室記録なし'})
    df['Embarked'] = df['Embarked'].fillna('不明')
    df['FareQuartile'] = pd.qcut(df['Fare'], 4, labels=['Q1低','Q2','Q3','Q4高'])
    df['Female'] = (df['Sex']=='female').astype(int)
    overview = eda.explore(df_raw)
    print('EDA報告項目:', list(vars(overview)))
    print('列型・欠損:', pd.DataFrame({'dtype':df_raw.dtypes.astype(str),'missing_n':df_raw.isna().sum(),'missing_pct':df_raw.isna().mean()*100}).to_string())
    print('数値要約:',df_raw[['Age','Fare','SibSp','Parch']].describe().to_string())
    print('元データのレコード削除:0、年齢/客室欠損の補完:0、港コード不明の分類:2。')
    print('運賃ゼロ件数:',int(df['Fare'].eq(0).sum()), '運賃最大:',df['Fare'].max(), '同一Ticket複数人のレコード数:',int(df['Ticket'].duplicated(keep=False).sum()))
''')
md('''
## 初回比較：率、分布、効果量
率の分母は各群の全レコード。連続変数の要約のみ当該変数の観測値を使う。Wilson区間は95%の点別区間で、群間差の区間や多重比較の同時区間ではない。Mann–Whitneyは平均差の検定ではなく順位分布の比較である。
''')
code('''
with execution_phase('初回群間比較'):
    def wilson(k, n):
        z = stats.norm.ppf(0.975)
        p = k/n
        den = 1 + z*z/n
        center = (p + z*z/(2*n))/den
        half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
        return center-half, center+half

    def rate_table(data, columns):
        tab = data.groupby(columns, observed=True, dropna=False)['Survived'].agg(n='size',survivors='sum').reset_index()
        tab['rate_pct'] = 100*tab['survivors']/tab['n']
        cis = [wilson(int(k),int(n)) for k,n in zip(tab['survivors'],tab['n'])]
        tab['ci_low_pct'] = [100*x[0] for x in cis]
        tab['ci_high_pct'] = [100*x[1] for x in cis]
        return tab

    total = rate_table(df.assign(Group='全体'), ['Group'])
    print('全体:', total.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
    tables = {}
    tests = []
    for col in ['Sex','Pclass','AgeGroup','FareQuartile','FamilyGroup','Embarked','AgeMissing','CabinKnown']:
        tables[col] = rate_table(df,[col])
        print('\\n群別:', col)
        print(tables[col].to_string(index=False, float_format=lambda x:f'{x:.6f}'))
        chi, p, dof, expected = stats.chi2_contingency(pd.crosstab(df[col],df['Survived']),correction=False)
        tests.append({'comparison':col,'test':'chi-square','p_raw':p,'effect':math.sqrt(chi/len(df)),'effect_name':'Cramers V','expected_min':expected.min()})
    continuous_rows=[]
    for col in ['Age','Fare']:
        a = df.loc[df['Survived']==1,col].dropna()
        b = df.loc[df['Survived']==0,col].dropna()
        for label, values in [('生存',a),('非生存',b)]:
            continuous_rows.append({'variable':col,'group':label,'n':len(values),'mean':values.mean(),'median':values.median(),'q25':values.quantile(.25),'q75':values.quantile(.75)})
        u,p=stats.mannwhitneyu(a,b,alternative='two-sided')
        tests.append({'comparison':col,'test':'Mann-Whitney','p_raw':p,'effect':2*u/(len(a)*len(b))-1,'effect_name':'rank-biserial survival minus non-survival','expected_min':np.nan})
    continuous = pd.DataFrame(continuous_rows)
    test_table = pd.DataFrame(tests)
    test_table['p_holm'] = multipletests(test_table['p_raw'],method='holm')[1]
    print('\\n連続変数要約:',continuous.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
    print('\\n探索的検定（Holm補正10比較）:',test_table.to_string(index=False,float_format=lambda x:f'{x:.8g}'))
    marginal_correlation = stats_analysis.correlation(df,'Female','Survived',language='ja')
    print('点双列相関:',marginal_correlation)
    print('女性と男性の粗生存率差pp:',100*(df.loc[df.Female==1,'Survived'].mean()-df.loc[df.Female==0,'Survived'].mean()))
    print('1等と3等の粗生存率差pp:',100*(df.loc[df.Pclass==1,'Survived'].mean()-df.loc[df.Pclass==3,'Survived'].mean()))
''')
md('''
### 初回図表の読み方
性別・等級・年齢・運賃四分位・同行家族の率を表示する。すべて割合で比較し、人数の違いは上の表で確認する。性別と生存の相関は二値同士の周辺関連であり、等級や年齢を調整した効果ではない。運賃の四分位境界はこの標本内の相対的分類で、通貨や1人当たり価格に関する主張には使わない。
''')
code('''
with execution_phase('初回図表'):
    chart(tables['Sex'], 'Sex', 'rate_pct', '性別ごとの生存率', '記録された性別', '生存率（%）')
    chart(tables['Pclass'].assign(Pclass=tables['Pclass']['Pclass'].astype(str)), 'Pclass', 'rate_pct', '客室等級ごとの生存率', '等級', '生存率（%）')
    age_plot = tables['AgeGroup'].set_index('AgeGroup').reindex(['0–11歳','12–17歳','18–34歳','35–59歳','60歳以上','年齢不明']).reset_index()
    chart(age_plot, 'AgeGroup', 'rate_pct', '年齢層ごとの生存率（欠損を別群）', '年齢層', '生存率（%）')
    chart(tables['FareQuartile'], 'FareQuartile', 'rate_pct', '運賃四分位ごとの生存率', '標本内の運賃四分位', '生存率（%）')
    chart(tables['FamilyGroup'], 'FamilyGroup', 'rate_pct', '同行家族人数指標ごとの生存率', '本人を含む人数指標', '生存率（%）')
''')

def replace_initial(nb):
    nb.cells = cells
    nb.metadata['analysis_run'] = {'run_id':'v020-exp-01','execution_route':'Jupyter MCP','language':'ja'}
lifecycle.register_run('v020-exp-01', handle.notebook_path)
lifecycle.mark_write_start('v020-exp-01')
try:
    pm.enqueue_write(handle, replace_initial)
finally:
    lifecycle.mark_write_end('v020-exp-01')
print('初回分析Notebookをenqueue_writeで構成:',len(cells),'cells')

初回分析Notebookをenqueue_writeで構成: 10 cells
